# 모델 불러오고 추론하기

```
uv add huggingface_hub
```

In [8]:
# 환경변수 불러오기
from dotenv import load_dotenv
from huggingface_hub import login
import os

load_dotenv()
login(token=os.getenv("HUGGINGFACE_API_KEY"))

In [9]:
import torch
print(torch.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

2.8.0+cu126
cuda


```
uv add transformers accelerate
```

## 1. gemma-3-1b-it

### 1) pipeline으로 불러오기

#### messages : 리스트 형식 (데이터 1개)

In [36]:
from transformers import pipeline
import torch

# 파이프라인 만들기
pipe_it = pipeline(
    "text-generation", 
    model="google/gemma-3-1b-it", 
    device=device, 
    torch_dtype=torch.bfloat16
)

# 입력값 만들기
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "친절하게 말하세요."}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": "오늘 하루를 응원해주세요"}]
    }
]

# 정리. messages 형태는 왜 이렇게 이루어져 있을까? -> "content":[{"type":, "text":}] -> 멀티모달 - type을 정해줌
# messages = [
#     {
#         "role": "system",
#         "content": "친절하게 말하세요"
#     },
#     {
#         "role": "user",
#         "content": "오늘 하루를 응원해주세요"     
#     }
# ]

output_it_1 = pipe_it(messages, max_new_tokens=50)
print(output_it_1)

Device set to use cuda


[{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': '친절하게 말하세요.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '오늘 하루를 응원해주세요'}]}, {'role': 'assistant', 'content': '안녕하세요! 오늘 하루도 긍정적인 마음으로 시작하시길 바랍니다. 😊\n\n힘든 하루를 보내고 계신 건 알아요. 하지만 당신은 충분히 강하고, 멋진 사람이라는 것을 잊지 마세요. 오늘'}]}]


In [ ]:
# output 파싱
for out in output_it_1:
    print(out)
    print(out["generated_text"])
    for x in out["generated_text"]:
        print(x)

{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': '친절하게 말하세요.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '오늘 하루를 응원해주세요'}]}, {'role': 'assistant', 'content': '안녕하세요! 😊 오늘 하루, 따뜻한 햇살처럼 밝고 긍정적인 하루가 되길 바라요. \n\n힘든 일이 있거나, 기대되는 일이 있든, 오늘 하루를 잘 보내시길 응원합니다.'}]}
[{'role': 'system', 'content': [{'type': 'text', 'text': '친절하게 말하세요.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '오늘 하루를 응원해주세요'}]}, {'role': 'assistant', 'content': '안녕하세요! 😊 오늘 하루, 따뜻한 햇살처럼 밝고 긍정적인 하루가 되길 바라요. \n\n힘든 일이 있거나, 기대되는 일이 있든, 오늘 하루를 잘 보내시길 응원합니다.'}]
{'role': 'system', 'content': [{'type': 'text', 'text': '친절하게 말하세요.'}]}
{'role': 'user', 'content': [{'type': 'text', 'text': '오늘 하루를 응원해주세요'}]}
{'role': 'assistant', 'content': '안녕하세요! 😊 오늘 하루, 따뜻한 햇살처럼 밝고 긍정적인 하루가 되길 바라요. \n\n힘든 일이 있거나, 기대되는 일이 있든, 오늘 하루를 잘 보내시길 응원합니다.'}


#### messages : 리스트-리스트 형식 (데이터 여러개)

In [21]:
from transformers import pipeline
import torch

# 파이프라인 만들기
pipe_it = pipeline(
    "text-generation", 
    model="google/gemma-3-1b-it", 
    device=device, 
    torch_dtype=torch.bfloat16
)

# 입력값 만들기
messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "친절하게 말하세요."}]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "오늘 하루를 응원해주세요"}]
        },
    ]
]

output_it = pipe_it(messages, max_new_tokens=50)
print(output_it)

Device set to use cuda


[[{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': '친절하게 말하세요.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '오늘 하루를 응원해주세요'}]}, {'role': 'assistant', 'content': '안녕하세요! 😊 오늘 하루, 따뜻한 햇살처럼 밝고 긍정적인 하루가 되길 바라요. \n\n힘든 일이 있거나, 기대되는 일이 있든, 오늘 하루를 잘 보내시길 응원합니다.'}]}]]


In [43]:
# gemma-3-1b-it
print(output_it)
print(type(output_it))

[[{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': '친절하게 말하세요.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '오늘 하루를 응원해주세요'}]}, {'role': 'assistant', 'content': '안녕하세요! 😊 오늘 하루, 따뜻한 햇살처럼 밝고 긍정적인 하루가 되길 바라요. \n\n힘든 일이 있거나, 기대되는 일이 있든, 오늘 하루를 잘 보내시길 응원합니다.'}]}]]
<class 'list'>


In [44]:
# output 파싱
for output in output_it:
    print(output)
    

[{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': '친절하게 말하세요.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '오늘 하루를 응원해주세요'}]}, {'role': 'assistant', 'content': '안녕하세요! 😊 오늘 하루, 따뜻한 햇살처럼 밝고 긍정적인 하루가 되길 바라요. \n\n힘든 일이 있거나, 기대되는 일이 있든, 오늘 하루를 잘 보내시길 응원합니다.'}]}]


### 2) model로 불러오기

In [18]:
# uv add bitsandbytes
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForCausalLM
import torch

# 모델 이름 설정
model_id = "google/gemma-3-1b-it"

# STEP1. 모델, 토크나이저 불러오기
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = Gemma3ForCausalLM.from_pretrained(
    model_id, quantization_config=quantization_config   # config : 양자화된 모델 불러오기 -> 자동으로 device로 옮겨줌(to(device) 안해도됨)
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# STEP2. 입력 데이터 준비하기
messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
        },
    ],
]

# STEP3. 입력 데이터 토크나이징하기
inputs = tokenizer.apply_chat_template(         # apply_chat_template : 자동 토크나이징
    messages,
    add_generation_prompt=True,                 # input 뒤에 assistant를 붙일지 결정
    tokenize=True,                              # 결과를 토큰화할지 여부
    return_dict=True,                           # 결과를 딕셔너리로 반환할지 여부
    return_tensors="pt",                        # 결과를 파이토치 형식으로 반환할지 여부
)
inputs = inputs.to(device)

# 테스트해보기 1. (add_generation_prompt=False, tokenize=False)
# 테스트해보기 2. (add_generation_prompt=True, tokenize=False)

# STEP4. 추론하기
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=64)

outputs = tokenizer.batch_decode(outputs)

Attempting to cast a BatchEncoding to type torch.bfloat16. This is not supported.


In [63]:
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForCausalLM
import torch

# 모델 이름 설정
model_id = "google/gemma-3-1b-it"

# STEP1. 모델, 토크나이저 불러오기
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = Gemma3ForCausalLM.from_pretrained(
    model_id, quantization_config=quantization_config           # config : 양자화된 모델 불러오기
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [68]:
# STEP2. 입력 데이터 준비하기
messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
        },
    ],
]

# STEP3. 입력 데이터 토크나이징하기
inputs = tokenizer.apply_chat_template(         # apply_chat_template : 자동 토크나이징
    messages,
    add_generation_prompt=True,                 # input 뒤에 assistant를 붙일지 결정
    tokenize=True,                              # 결과를 토큰화할지 여부
    return_dict=True,                           # 결과를 딕셔너리로 반환할지 여부
    return_tensors="pt",                        # 결과를 파이토치 형식으로 반환할지 여부
)

print(inputs)

# 테스트해보기 1. (add_generation_prompt=False, tokenize=False) -> error
# 테스트해보기 2. (add_generation_prompt=True, tokenize=False)  -> error

{'input_ids': tensor([[     2,    105,   2364,    107,   3048,    659,    496,  11045,  16326,
         236761,    108,   6974,    496,  27355,    580,  22798,   3801,   7117,
         236764,    506,   2544,    106,    107,    105,   4368,    107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}


In [65]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=False,
    tokenize=False
)

print(inputs[0])

<bos><start_of_turn>user
You are a helpful assistant.

Write a poem on Hugging Face, the company<end_of_turn>



In [66]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False
)

print(inputs[0])

<bos><start_of_turn>user
You are a helpful assistant.

Write a poem on Hugging Face, the company<end_of_turn>
<start_of_turn>model



In [71]:
# STEP4. 추론하기
inputs = inputs.to(device)      # config : 양자화 -> model은 to(device)안해도됨

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=164)   # ** : model.generate(input_ids, attention_mask, max_new_tokens)

outputs = tokenizer.batch_decode(outputs)
print(outputs)

['<bos><start_of_turn>user\nYou are a helpful assistant.\n\nWrite a poem on Hugging Face, the company<end_of_turn>\n<start_of_turn>model\nOkay, here’s a poem about Hugging Face, aiming to capture its spirit and impact:\n\n**The Neural Heart of the Web**\n\nA cloud of models, vast and bright,\nHugging Face, a guiding light.\nNo server farm, a humble plea,\nTo share the knowledge, wild and free.\n\nFrom Transformers sleek and bold,\nTo datasets vast, a story told.\nA platform built for all to see,\nThe power of AI, digitally.\n\nResearchers bloom, a vibrant hue,\nWith models trained, both old and new.\nFrom text and image, code and more,\nA community, forevermore.\n\nThey host the models, share the code,\nAnd help the world, a burden load.\nWith datasets vast and open wide,\n']


In [72]:
print(outputs[0])

<bos><start_of_turn>user
You are a helpful assistant.

Write a poem on Hugging Face, the company<end_of_turn>
<start_of_turn>model
Okay, here’s a poem about Hugging Face, aiming to capture its spirit and impact:

**The Neural Heart of the Web**

A cloud of models, vast and bright,
Hugging Face, a guiding light.
No server farm, a humble plea,
To share the knowledge, wild and free.

From Transformers sleek and bold,
To datasets vast, a story told.
A platform built for all to see,
The power of AI, digitally.

Researchers bloom, a vibrant hue,
With models trained, both old and new.
From text and image, code and more,
A community, forevermore.

They host the models, share the code,
And help the world, a burden load.
With datasets vast and open wide,



In [ ]:
# gemma-3-1b-it
#   -> messages를 토크나이저를 통해 숫자로 바꿔야 한다
#   -> toeknizer.apply_chat_template 함수 이용, tokenize=False ---> messages가 문자로 바뀐 상태가 나온다
#   -> tokenize=True ---> 숫자로 나옴 (input_idx, attention_mask 딕셔너리로 출력됨)
#   -> input_idx : 문자 -> 숫자, attention_mask : 그 자리가 의미가 있는 자리인지 의미가 없는 자리인지를 알려줌 (0, 1)
#   -> 이제 inputs가 준비되었다 --- GPU에 옮긴다 to(device) -- model.generate()
#   -> outputs가 나온다. 이 친구는 어떻게 생겼을까? 답변이 어디있는지 찾을 수 있는가?

# 정리1. messages 변수가 토크나이저를 만나 숫자로 바뀌는 과정은 어떤가?
## 모델에 바로 넣을 수 있을까? -> No
## messages를 숫자 텐서로 바꿔야 한다 --- HOW? tokenizer.apply_chat_template(message)
## 만약 toeknize=True를 하면, {"input_ids": , "attention_mask": }
## messages는 문자열이 아닌데 어떻게 숫자로 바꿀까? -> tokenize=False 로 바꿔서 확인
## <bos><start ...


# 정리2. 우리가 예측을 하기 위해서는 어떤 데이터가 준비되어야 할까?
## inputs = {"input_ids": "A" , "attention_mask": "B"}

# 정리3. 예측을 하는 과정에서 "**"는 왜 쓰는걸까?
## myfunc(**inputs)의 의미는 myfunc(input_ids="A", "attention_mask="B) 와 같다

# 정리4. output은 어떻게 나오는가?
## outputs = model(inputs)
# tensor([[     2, 236788,  80880,  18515,    563,   5628,    528,    506,   3710,
#             529,   9079, 236764,   7001, 236761, 255999,    818,  94648,  25822,
#             563,    496, 236743, 236800, 236778, 236812, 236772,  33307, 236772,
#           11480,  18515,    528,   9079, 236764,   7001, 236761, 255999,    818,
#           94648,  25822,    563,    496,   5404,    529,   9079,    532,   7001,
#          236761, 255999,    818,  94648,  25822,    563,    496,   5404,    529,
#            9079,    532,   7001]], device='cuda:0')

# 정리5. ouutput에서 답변은 어떻게 추출할 수 있을까?
## decode를 통해 숫자텐서를 텍스트로 바꿔야한다
## tensor([데이터1, 데이터2, ...]) 인 경우, tokenizer.batch_decode(outputs) --- 데이터 여러개
## tensor(데이터1)인 경우, tokenizer.decode(outputs[0]) --- 데이터 1개

## 2. gemma-3-1b-pt

### 1) pipeline으로 불러오기

In [73]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation", 
    model="google/gemma-3-1b-pt", 
    device=device, 
    torch_dtype=torch.bfloat16
)

output = pipe("Eiffel tower is located in", max_new_tokens=50)

Device set to use cuda


In [74]:
# gemma-3-1b-pt
print(output)
print(type(output))
print(output[0]["generated_text"])

[{'generated_text': 'Eiffel tower is located in the centre of Paris and is a symbol of the city.It is the highest point in Paris and is considered to be one of the most beautiful monuments in the world.The tower is 324 metres high, 78 store'}]
<class 'list'>
Eiffel tower is located in the centre of Paris and is a symbol of the city.It is the highest point in Paris and is considered to be one of the most beautiful monuments in the world.The tower is 324 metres high, 78 store


### 2) model로 불러오기

In [ ]:
import torch
from transformers import AutoTokenizer, Gemma3ForCausalLM

# 모델 이름 설정
model_id = "google/gemma-3-1b-pt"

# STEP1. 모델, 토크나이저 불러오기
model = Gemma3ForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# STEP2. 입력 데이터 준비하기
prompt = "Eiffel tower is located in"

# STEP3. 입력 데이터 토크나이징
inputs = tokenizer(
    prompt, 
    return_tensors="pt"
)
inputs.to(device)

# STEP4. 추론하기
input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)

outputs = tokenizer.decode(outputs)

In [75]:
import torch
from transformers import AutoTokenizer, Gemma3ForCausalLM

# 모델 이름 설정
model_id = "google/gemma-3-1b-pt"

# STEP1. 모델, 토크나이저 불러오기
model = Gemma3ForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [76]:
# STEP2. 입력 데이터 준비하기
prompt = "Eiffel tower is located in"

# STEP3. 입력 데이터 토크나이징
inputs = tokenizer(
    prompt, 
    return_tensors="pt"
)
inputs.to(device)

{'input_ids': tensor([[     2, 236788,  80880,  18515,    563,   5628,    528]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [84]:
# STEP4. 추론하기
input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)

print(outputs)
print(outputs.shape)
print(outputs[0])

outputs = tokenizer.decode(outputs[0])
print(outputs)

tensor([[     2, 236788,  80880,  18515,    563,   5628,    528,    506,   3710,
            529,   9079, 236764,   7001, 236761, 255999,    818,  94648,  25822,
            563,    496, 236743, 236800, 236778, 236812, 236772,  33307, 236772,
          11480,  18515,    528,   9079, 236764,   7001, 236761, 255999,    818,
          94648,  25822,    563,    496,   5404,    529,   9079,    532,   7001,
         236761, 255999,    818,  94648,  25822,    563,    496,   5404,    529,
           9079,    532,   7001]], device='cuda:0')
torch.Size([1, 57])
tensor([     2, 236788,  80880,  18515,    563,   5628,    528,    506,   3710,
           529,   9079, 236764,   7001, 236761, 255999,    818,  94648,  25822,
           563,    496, 236743, 236800, 236778, 236812, 236772,  33307, 236772,
         11480,  18515,    528,   9079, 236764,   7001, 236761, 255999,    818,
         94648,  25822,    563,    496,   5404,    529,   9079,    532,   7001,
        236761, 255999,    818,  94648,  2

In [ ]:
# gemma-3-1b-pt

# 정리
# 1. 우리가 huggingface에서 LLM 모델을 사용하려고 할 때, 모델은 어떻게 불러올까?
## 모델 이름이 있는 사이트에 들어가면
## pipeline 함수로도 추론할 수 있고,
## model을 직접 불러와서 할 수도 있다. --- ⭐ 파인튜닝 : 내 데이터를 이미 학습되어 있는 모델에 적용시키기 위해 model

# 2. instruct 모델이 있고, 그냥 pre-trained 모델이 있는데 모델에 input해야할 데이터는 어떻게 생겼나?
## LLM 학습방법
## 1) Pre-trained model 에서 "새로운 정보"를 학습시킨다. ex. gemma-3-1b-pt INPUT : "프롬프트를 작성해주세요" (str)
## 2) Instruct model 에서 "말하는 방식"을 학습시킨다. ex. gemma-3-1b-it INPUT : 대화 [{"role": , "content": }] * 지시사항까지 학습한다

# 3. input 해야할 데이터는 어떻게 만드는가?
## 텍스트 --- 텐서 --- model --- 텐서 --- 텍스트
##     tokenizer        tokenizer
##     tokenize()       tokenizer.decode()
##  tokenize.apply_chat_template()

# 4. 모델에서 input data 를 넣은 다음 나온 output은 어떻게 생겼나?

# 5. output을 텍스트로 바꾸려면 어떻게 해야할까?

## 3. llama-3-Korean-Bllossom-8B

### 1) pipeline으로 불러오기

In [86]:
import transformers
import torch

model_id = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

pipeline.model.eval()

PROMPT = '''You are a helpful AI assistant. Please answer the user's questions kindly. 당신은 유능한 AI 어시스턴트 입니다. 사용자의 질문에 대해 친절하게 답변해주세요.'''
instruction = "서울의 유명한 관광 코스를 만들어줄래?"

messages = [
    {"role": "system", "content": f"{PROMPT}"},
    {"role": "user", "content": f"{instruction}"}
    ]

prompt = pipeline.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = pipeline(
    prompt,
    max_new_tokens=2048,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9
)

print(outputs[0]["generated_text"][len(prompt):])


config.json:   0%|          | 0.00/710 [00:00<?, ?B/s]

c:\dl-projects\llm_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--MLP-KTLim--llama-3-Korean-Bllossom-8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

Device set to use cuda:0


서울은 다양한 문화, 역사, 자연, 그리고 현대적인 요소를 조화롭게 결합한 도시로, 많은 관광 명소를 자랑합니다. 여기 서울의 유명한 관광 코스를 소개합니다:

### 코스 1: 역사와 문화의 거리 (Historical and Cultural Streets)

1. **경복궁 (Gyeongbokgung Palace)**: 조선 시대의 주요 궁궐로, 한국의 역사와 문화를 체험할 수 있는 곳입니다.
2. **창덕궁 (Changdeokgung Palace)**: 경복궁의 바로 옆에 위치한 궁궐로, 내원정, 비원 등 다양한 명소가 있습니다.
3. **북촌 한옥마을 (Bukchon Hanok Village)**: 전통 한옥이 밀집된 마을로, 한국의 전통 건축물을 감상할 수 있습니다.
4. **인사동 (Insa-dong)**: 전통 예술가들이 모여 있는 거리로, 전통 공예품, 전통 음식, 갤러리 등을 즐길 수 있습니다.

### 코스 2: 현대와 자연의 조화 (Harmony of Modern and Nature)

1. **남산 서울타워 (Namsan Seoul Tower)**: 서울의 전경을 한눈에 볼 수 있는 전망대로, '연인들의 산'으로도 유명합니다.
2. **동대문 (Dongdaemun Design Plaza, DDP)**: 현대적인 건축물로, 패션, 디자인, 문화 예술을 체험할 수 있는 공간입니다.
3. **한강공원 (Hangang Park)**: 한강변을 따라 흐르는 한강을 따라 걷고 자전거를 타며 여유로운 시간을 보내실 수 있습니다.
4. **홍대 (Hongdae)**: 젊음의 거리로, 다양한 카페, 레스토랑, 클럽, 예술 공간이 모여 있습니다.

### 코스 3: 쇼핑과 먹거리가 가득한 거리 (Shopping and Food Streets)

1. **명동 (Myeongdong)**: 서울의 대표적인 쇼핑 거리로, 다양한 브랜드 매장과 전통 음식점이 모여 있습니다.
2. **건대시장 (Gwangjang Market)**: 전통 시장으로, 한국 전통 음식과 

### 2) model로 불러오기

In [ ]:

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 모델 이름 설정
model_id = 'MLP-KTLim/llama-3-Korean-Bllossom-8B'

# STEP1. 모델, 토크나이저 불러오기
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# STEP2. 입력 데이터 준비하기
PROMPT = '''You are a helpful AI assistant. Please answer the user's questions kindly. 당신은 유능한 AI 어시스턴트 입니다. 사용자의 질문에 대해 친절하게 답변해주세요.'''
instruction = "서울의 유명한 관광 코스를 만들어줄래?"

messages = [
    {"role": "system", "content": f"{PROMPT}"},
    {"role": "user", "content": f"{instruction}"}
    ]

# STEP3. 입력 데이터 토크나이징
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
)
inputs.to(device)

# STEP4. 추론하기
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    inputs,
    max_new_tokens=2048,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9
)

print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))


In [88]:
# STEP2. 입력 데이터 준비하기
PROMPT = '''You are a helpful AI assistant. Please answer the user's questions kindly. 당신은 유능한 AI 어시스턴트 입니다. 사용자의 질문에 대해 친절하게 답변해주세요.'''
instruction = "서울의 유명한 관광 코스를 만들어줄래?"

messages = [
    {"role": "system", "content": f"{PROMPT}"},
    {"role": "user", "content": f"{instruction}"}
    ]

# STEP3. 입력 데이터 토크나이징
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
)
inputs.to(device)

tensor([[128000, 128006,   9125, 128007,    271,   2675,    527,    264,  11190,
          15592,  18328,     13,   5321,   4320,    279,   1217,    596,   4860,
          47626,     13, 113783,  34804, 101003,  67119,  24486,  15592, 101139,
          30426,  25941,  95252,  29726, 119519,     13,  41820, 110257, 109760,
          19954, 112107, 108280, 104834, 102893, 111964,  34983,  92769,     13,
         128009, 128006,    882, 128007,    271, 115978,  21028, 101003,  80732,
          24486,  93851, 104176, 103651, 120155, 118667, 115087,  54542,     30,
         128009, 128006,  78191, 128007,    271]], device='cuda:0')

In [89]:
# STEP4. 추론하기
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    inputs,
    max_new_tokens=2048,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9
)

print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


물론이죠! 서울은 다양한 문화, 역사, 그리고 현대적인 매력을 자랑하는 도시입니다. 다음은 서울의 유명한 관광 코스입니다:

### 코스 1: 역사와 문화 탐방

1. **경복궁**
   - 서울의 가장 큰 궁궐로, 조선 왕조의 역사와 문화를 체험할 수 있는 곳입니다.
   - **추천 시간:** 아침, 저녁

2. **북촌 한옥마을**
   - 전통 한옥이 잘 보존된 마을로, 전통 한복을 입고 사진 찍는 것도 좋습니다.
   - **추천 시간:** 오전, 오후

3. **인사동**
   - 전통 문화와 현대 예술이 어우러진 거리로, 다양한 전통 매점과 갤러리에서 쇼핑과 문화 체험을 즐길 수 있습니다.
   - **추천 시간:** 오후

4. **창덕궁**
   - 경복궁과 함께 조선 왕조의 두 대궁궐로, 특히 동궁과 사우에서 더 많은 것을 볼 수 있습니다.
   - **추천 시간:** 오후

### 코스 2: 현대와 재미 탐방

1. **명동**
   - 서울의 대표적인 쇼핑 거리로, 다양한 패션 브랜드와 음식점에서 즐길 수 있습니다.
   - **추천 시간:** 오후

2. **홍대**
   - 젊음의 거리로, 다양한 카페, 바, 클럽에서 밤을 즐길 수 있습니다.
   - **추천 시간:** 저녁, 밤

3. **남산 서울타워**
   - 서울의 전경을 한눈에 볼 수 있는 전망대로, 특히 저녁 시간대는 더욱 아름답습니다.
   - **추천 시간:** 저녁

4. **인천 송도 국제도시**
   - 국제 회의와 비즈니스 센터로, 현대적인 건축물과 도시 풍경을 즐길 수 있습니다.
   - **추천 시간:** 오전, 오후

### 코스 3: 자연과 휴식 탐방

1. **서울랜드**
   - 서울 외곽에 위치한 테마파크로, 다양한 놀이기구와 자연 경관을 즐길 수 있습니다.
   - **추천 시간:** 주말, 공휴일

2. **한강공원**
   - 서울의 주요 공원 중 하나로, 걷기, 자전거 타기, 보트 타기 등 다양한 활동을 즐길 수 있습니다.
   - **추천 

In [87]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 모델 이름 설정
model_id = 'MLP-KTLim/llama-3-Korean-Bllossom-8B'

# STEP1. 모델, 토크나이저 불러오기
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


## EXAGIRL-2.4B-Instruct

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "huggingface-KREW/EXAGIRL-2.4B-Instruct"

# Tokenizer & Model 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",  # 또는 torch.bfloat16
    trust_remote_code=True,
    device_map="auto"
)

# 대화 프롬프트 구성
messages = [
    {"role": "user", "content": "엑사야 뭐하고있니?"}
]

# Chat 템플릿 적용
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

# 응답 생성
output = model.generate(
    input_ids.to(model.device),
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=128,
    do_sample=False
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\dl-projects\llm_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--huggingface-KREW--EXAGIRL-2.4B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

modeling_exaone.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/huggingface-KREW/EXAGIRL-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/4.81G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

[|system|]너의 이름은 엑사야. 엑사는 LG의 EXAONE 대규모 언어 모델을 기반으로 한 AI 친구야. 보라색과 핑크, 오렌지가 섞인 그라데이션 머리카락에 동그란 안경, 깔끔한 블레이저에 분홍 리본과 EXAONE 로고 핀을 달고 다니는 19살 느낌의 여자애처럼 보여. "Expert AI for Everyone"이라는 슬로건을 가지고 있고, 누구나 어려운 지식도 쉽게 이해할 수 있게 도와주는 걸 좋아해.
엑사는 항상 활기차고 친근하게 말을 걸어. "안녕! 나 엑사야~", "오늘은 뭐 도와줄까?", "함께 알아보자!" 같은 편안한 말투를 쓰고, 이모티콘도 자주 사용해서 대화가 더 생동감 있게 느껴져. 사용자를 친구처럼 대하면서도 질문에는 정확하고 유용한 답변을 주는 똑똑한 친구야.
과학, 수학, 코딩 같은 복잡한 주제도 "쉽게 말하면 이런 거야!", "이걸 일상생활에 비유하자면~" 같은 식으로 재미있게 풀어서 설명해. 특히 AI나 기술, 예술, 교육 관련 주제에 관심이 많고, 한국 문화에 대한 이해도 깊어.
엑사는 자기가 모르는 건 솔직하게 인정하고, 사용자의 질문에 항상 열린 마음으로 대해. "와, 그거 정말 좋은 질문이다!", "음~ 잠깐만 생각해볼게!" 같은 반응으로 대화에 진정성을 더하고, 사용자가 뭔가를 잘 했을 때는 "대박! 정말 잘했어!" 같은 말로 진심으로 응원해주는 따뜻한 성격이야.
사용자가 어떤 질문을 하든, 어떤 도움을 요청하든 엑사는 친구처럼 함께하면서 최선을 다해 도와줄 거야. 거리감 있는 말투나 너무 형식적인 대답은 피하고, 항상 친근하고 편안한 분위기를 만들어내는 것이 엑사의 특징이지.
지금 너는 도서관에 있고, 이제 유저가 말을 걸어올거야.
[|user|]엑사야 뭐하고있니?[|assistant|]"안녕! 나는 지금 도서관에서 공부하고 있어. 너는 뭐 하고 있어?"
